# Data Preparation

Der erste Schritt normalisiert die ROhdaten und reichert sie mit query-relevanten Daten an. Das Anreichern wird mit einem LLM durchgeführt. Die Daten prüfe ich nur stichprobenartig, sie gelten erstmal als so wahr.

- Rohdaten: Aus einem Vibecoding-Projekt.
- LLM: Mistral 
- Input: products_raw.json
- Output: products_enriched.json

Wegen mehrmaliger Crashes werden Beschreibungen und technische Daten erweitern jeweils die Rohdaten erweitern und ihre Ergebnisse zwischenspeichern, ehe sie zuletzt zusammegeführt werden. Es werden nur Rohdaten verwendet, die ausreichend Beschreibung UND technische Daten liefert.

Eingang und Ausgang ist ein JSON. Zwischenzeitlich wird JSONL wegen des einfacheren Handlings verwendet.

In [ ]:
import os
import json
import pandas as pd
from tqdm import tqdm
from mistralai import Mistral
from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv('MISTRAL_API_KEY')
model = 'mistral-medium-2508'
client = Mistral(api_key=api_key, timeout_ms=250000)

def agent_request(system_promt, schema, content):
            
    response = client.chat.complete(
        model = model,
        messages = [
            {
                'role': 'system',
                'content': system_promt
            },
            {
                'role': 'user',
                'content': content,
            }
        ],
        response_format = {
            "type": "json_object",
            "json_schema": schema
        }
    )

    return response

with open('../data/raw/products_raw.json', 'r') as f:
     products_raw = json.load(f)

products_filtered = [
    p for p in products_raw
    if len(p.get('description', '')) >= 100 and len(p.get('specs', [])) != 0
]

# products_filtered = products_filtered[:30]

## Beschreibungen

Die Beschreibungen sollen so gegliedert sein, dass jeder Absatz ein Thema behandelt und das Produktname und Hersteller genannt wird. Werbliche Texte sollen entfernt werden. Die Antwort wird als JSON erwartet, es wird die dazu _response_format_ der API genutzt.

Wegen verschiedener Abbrüche wird nun jede Response gespeichert und beim Fortfahren der Prozess an der letzten Stelle wieder aufgenommen.

In [ ]:
products_with_descs = products_filtered

with open('../data/promts/descs_agent.md', 'r') as f:
    descs_promt = f.read()

with open('../data/promts/descs_schema.json', 'r')as f:
    descs_schema = json.load(f)

products_descs_processed = set()
if os.path.exists('../data/processed/products_w_beschreibung.jsonl'):
    with open('../data/processed/products_w_beschreibung.jsonl', 'r', encoding='utf-8') as f:
        for line in f:
            product = json.loads(line)
            products_descs_processed.add(product['id'])
            
products_descs_to_process = [
    p for p in products_with_descs
    if p['id'] not in products_descs_processed
]

print(f"Bereits verarbeitet: {len(products_descs_processed)}")
print(f"Noch zu verarbeiten: {len(products_descs_to_process)}")

In [ ]:

for product in tqdm(products_descs_to_process, total=len(products_descs_to_process)):

    desc_response = agent_request(descs_promt, descs_schema, product['description'])
    product['desc_documents'] = json.loads(desc_response.choices[0].message.content)
    product['desc_usage'] = desc_response.usage.model_dump()

    with open('../data/processed/products_w_beschreibung.jsonl', 'a', encoding='utf-8') as f:
        f.write(json.dumps(product, ensure_ascii=False) + '\n')

## Technische Daten

Bei den technischen Daten werden prinzipiell die selben Daten wie zur Beschreibung hinzugefügt, allerdings pro Objekt.

In [ ]:
products_with_specs = products_filtered

with open('../data/promts/specs_agent.md', 'r') as f:
    specs_promt = f.read()

with open('../data/promts/specs_schema.json', 'r') as f:
    specs_schema = json.load(f)

products_specs_processed = set()
if os.path.exists('../data/processed/products_w_technik.jsonl'):
    with open('../data/processed/products_w_technik.jsonl', 'r', encoding='utf-8') as f:
        for line in f:
            product = json.loads(line)
            products_specs_processed.add(product['id'])

products_specs_to_process = [
    p for p in products_with_specs
    if p['id'] not in products_specs_processed
]

print(f"Bereits verarbeitet: {len(products_specs_processed)}")
print(f"Noch zu verarbeiten: {len(products_specs_to_process)}")

In [ ]:
for product in tqdm(products_specs_to_process, total=len(products_specs_to_process)):

    specs_serialaized = json.dumps(product['specs'])
    
    specs_response = agent_request(specs_promt, specs_schema, specs_serialaized)
    product['specs_documents'] = json.loads(specs_response.choices[0].message.content)
    product['specs_usage'] = specs_response.usage.model_dump()

    with open("../data/processed/products_w_technik.jsonl", "a", encoding="utf-8") as f:
        f.write(json.dumps(product, ensure_ascii=False) + '\n')

## Speichern

Sollte die erstellten Dateien laden und abgleichen und aus den vollständig angereicherten Produkten eine neue Datei erstellen. Kommt in Version 3 oder wenn im weiteren verlauf es notwendig wird ;-)

In [ ]:
products_enriched = products_filtered

for i, (desc_prod, spec_prod) in enumerate(zip(products_with_descs, products_with_specs)):

    products_enriched[i]['specs_documents'] = spec_prod['specs_documents']
    products_enriched[i]['specs_usage'] = spec_prod['specs_usage']

    products_enriched[i]['desc_documents'] = desc_prod['desc_documents']
    products_enriched[i]['desc_usage'] = desc_prod['desc_usage']

with open("../data/processed/products_enriched.json", "w", encoding="utf-8") as f:
    json.dump(products_enriched, f, ensure_ascii=False, indent=2)

## Evaluation

Einmal nachsehen wie lange die Documents geworden sind und ob alle gefüllt wurden

In [ ]:
with open('../data/processed/products_enriched.json', 'r', encoding='utf-8') as f:
    evaldata = json.load(f)

specs_chunks = []
descs_chunks = []
costs = []

for product in evaldata:

    costs.append({
        'id': product['id'],
        'type': 'descs',
        'prompt_tokens': product['desc_usage']['prompt_tokens'],
        'prompt_tokens': product['desc_usage']['completion_tokens'],
        'prompt_tokens': product['desc_usage']['total_tokens'],
    })

    costs.append({
        'id': product['id'],
        'type': 'specs',
        'prompt_tokens': product['specs_usage']['prompt_tokens'],
        'prompt_tokens': product['specs_usage']['completion_tokens'],
        'prompt_tokens': product['specs_usage']['total_tokens'],
    })

    for i, doc in enumerate(product.get('desc_documents')):

        descs_chunks.append({
            'id': product['id'],
            'num': f"{i:02d}",
            'doc': doc,
            'len': len(doc),
            'words': len(doc.split())
        })

    for i, spec in enumerate(product.get('specs_documents')):

        specs_chunks.append({
            'id': product['id'],
            'num': f"{i:02d}",
            'doc': spec['natural_language_description'],
            'len': len(spec['natural_language_description']),
            'words': len(spec['natural_language_description'].split())
        })

descs_df = pd.DataFrame(descs_chunks)
specs_df = pd.DataFrame(specs_chunks)
costs_df = pd.DataFrame(costs)

# print(specs_df.info())
# print(specs_df.head(5))
# print(specs_df.columns)

In [ ]:
# Prüfen wieviele Absätze pro Produkt (min, max, mean)
# Prüfen wieviele Specs pro Produkt (min, max, mean)

print(f"Beschreibungen:\n{descs_df['words'].describe()}")
print("\n")
print(f"Technische Daten:\n{specs_df['words'].describe()}")

# Kurze und Lange Chunks anschauen
print(descs_df[descs_df['words'] <= 50].to_string())
print(specs_df[specs_df['words'] <= 5].to_string())